# 01 — Data Exploration

This notebook walks through the raw data, cleaning pipeline, resampling, and exploratory data analysis (EDA) for the LOB alpha research project.

**Assets**: BTCUSDT, ETHUSDT (Binance Futures)  
**Data**: 10-level order book snapshots + tick-level trades (~2 hours)

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.utils.paths import load_config, get_data_dir
from src.data.load_data import load_raw_orderbook, load_raw_trades, load_processed
from src.data.clean_data import run_cleaning_pipeline
from src.data.resample_data import resample_and_align

config = load_config()
sns.set_theme(style='whitegrid', font_scale=1.1)
print('Config loaded. Assets:', config['assets'])

## 1. Load Raw Data

In [ ]:
raw = {}
for sym in config['assets']:
    ob = load_raw_orderbook(sym)
    tr = load_raw_trades(sym)
    raw[sym] = {'ob': ob, 'tr': tr}
    print(f"{sym}: {len(ob):,} OB snapshots, {len(tr):,} trades")
    print(f"  OB time range: {ob['timestamp'].min()} → {ob['timestamp'].max()}")
    print(f"  Trades time range: {tr['timestamp'].min()} → {tr['timestamp'].max()}")
    print()

## 2. Inspect Raw Order Book

In [ ]:
sym = 'BTCUSDT'
ob = raw[sym]['ob']
print(f"Order book columns ({len(ob.columns)}):")
print(ob.columns.tolist())
print(f"\nFirst 3 rows:")
ob.head(3)

In [ ]:
ob['spread'] = ob['ask_price_1'] - ob['bid_price_1']
ob['mid'] = (ob['ask_price_1'] + ob['bid_price_1']) / 2

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(ob['timestamp'], ob['mid'], linewidth=0.5)
axes[0].set_title(f'{sym} Mid-Price')
axes[0].set_ylabel('Price ($)')

axes[1].hist(ob['spread'], bins=50, edgecolor='black', alpha=0.7)
axes[1].set_title('Spread Distribution')
axes[1].set_xlabel('Spread ($)')

obi = (ob['bid_size_1'] - ob['ask_size_1']) / (ob['bid_size_1'] + ob['ask_size_1'])
axes[2].hist(obi, bins=80, edgecolor='black', alpha=0.7)
axes[2].set_title('Top-Level OBI')
axes[2].set_xlabel('OBI')

plt.tight_layout()
plt.show()

## 3. Inspect Raw Trades

In [ ]:
tr = raw[sym]['tr']
print(f"Trade columns: {tr.columns.tolist()}")
print(f"Trades per second: {len(tr) / ((tr['timestamp'].max() - tr['timestamp'].min()).total_seconds()):.1f}")
tr.head(3)

In [ ]:
tr_1s = tr.set_index('timestamp').resample('1s')['quantity'].sum()
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(tr_1s.index, tr_1s.values, linewidth=0.3, alpha=0.7)
axes[0].set_title(f'{sym} Volume per Second')
axes[0].set_ylabel('Volume')

if 'is_buyer_maker' in tr.columns:
    buy_pct = (~tr['is_buyer_maker']).mean() * 100
    axes[1].bar(['Buyer-initiated', 'Seller-initiated'],
                [buy_pct, 100 - buy_pct], color=['#2ecc71', '#e74c3c'])
    axes[1].set_title('Trade Aggressor Side')
    axes[1].set_ylabel('% of Trades')

plt.tight_layout()
plt.show()

## 4. Cleaning Pipeline

In [ ]:
cleaned = {}
for sym in config['assets']:
    ob = raw[sym]['ob']
    tr = raw[sym]['tr']
    clean_ob, clean_tr, reports = run_cleaning_pipeline(ob, tr, book_levels=10)
    cleaned[sym] = {'ob': clean_ob, 'tr': clean_tr, 'reports': reports}
    print(f"\n{sym} Cleaning Report (Order Book):")
    display(reports['orderbook'])
    print(f"\n{sym} Cleaning Report (Trades):")
    display(reports['trades'])

## 5. Resample & Align

In [ ]:
aligned = {}
for sym in config['assets']:
    df = resample_and_align(cleaned[sym]['ob'], cleaned[sym]['tr'], frequency='1s')
    aligned[sym] = df
    print(f"{sym}: {df.shape[0]:,} rows x {df.shape[1]} cols")
    print(f"  Time range: {df['timestamp'].min()} → {df['timestamp'].max()}")
    print(f"  Columns: {df.columns.tolist()[:10]}...")
    print()

## 6. Run Full EDA

This generates all 19 EDA figures specified in the project spec.

In [ ]:
from src.analysis.eda import run_full_eda

datasets = {sym: load_processed(sym, 'features') for sym in config['assets']}
saved = run_full_eda(datasets)
print(f"{len(saved)} EDA figures saved to reports/figures/")

## 7. Display Key EDA Figures

In [ ]:
from pathlib import Path
from IPython.display import Image, display as ipy_display

fig_dir = Path('..') / 'reports' / 'figures'
key_figs = [
    '01_spread_dist_btcusdt.png',
    '03_obi_dist_btcusdt.png',
    '04_return_by_obi_decile_btcusdt.png',
    '06_correlation_matrix_btcusdt.png',
    '07_signal_decay_btcusdt.png',
    '10_btc_vs_eth_comparison.png',
]

for fname in key_figs:
    path = fig_dir / fname
    if path.exists():
        print(f"\n--- {fname} ---")
        ipy_display(Image(filename=str(path), width=700))

## 8. Summary Statistics

In [ ]:
for sym in config['assets']:
    df = datasets[sym]
    cols = ['spread', 'relative_spread', 'obi_1', 'obi_5', 'weighted_obi',
            'microprice_deviation', 'trade_imbalance_1s', 'total_volume_1s',
            'return_lag_1s', 'realized_vol_10s',
            'y_return_1s', 'y_return_5s', 'y_return_10s', 'y_return_30s']
    cols = [c for c in cols if c in df.columns]
    print(f"\n{'='*60}")
    print(f"  {sym} Summary Statistics")
    print(f"{'='*60}")
    display(df[cols].describe().T.round(6))